## 1. Курс USD/RUB на вчерашний день (API ЦБ РФ)

In [1]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# официальный API ЦБ РФ отдаёт курс по конкретной дате. Берём вчерашний день,
# как указано в задании: "курс ЦБ на вчерашний день".
yesterday = datetime.now() - timedelta(days=1)
date_str = yesterday.strftime('%d/%m/%Y')

usd_rate = None
rate_date = None
rate_source = None

try:
    url = f"https://www.cbr.ru/scripts/XML_daily.asp?date_req={date_str}"
    resp = requests.get(url, timeout=5)
    resp.encoding = 'windows-1251'
    root = ET.fromstring(resp.content)

    for valute in root.findall('Valute'):
        if valute.find('CharCode').text == 'USD':
            nominal = int(valute.find('Nominal').text)
            value = float(valute.find('Value').text.replace(',', '.'))
            usd_rate = value / nominal
            break

    if usd_rate is None:
        raise ValueError("Валюта USD не найдена в ответе ЦБ РФ")

    # ЦБ возвращает дату НАЧАЛА действия курса, а не запрошенную дату:
    # по выходным новый курс не устанавливается, поэтому курс, объявленный
    # в пятницу, действует и в субботу, и в воскресенье, и в понедельник —
    # в архиве он хранится под пятничной/субботней датой начала действия.
    rate_date = root.attrib.get('Date', date_str)
    rate_source = "официальное API ЦБ РФ (XML_daily.asp)"
    print(f"Запрошена дата: {date_str} (вчера)")
    print(f"Курс USD, действовавший на эту дату (с {rate_source}): {usd_rate:.4f}")
    print(f"  Дата начала действия этого курса по данным ЦБ: {rate_date}")

except Exception as e_primary:
    # Резервный источник: JSON-зеркало того же курса ЦБ РФ
    print(f"Официальный API ЦБ РФ недоступен ({e_primary}), пробуем резервный источник...")
    try:
        resp = requests.get("https://www.cbr-xml-daily.ru/daily_json.js", timeout=5)
        data = resp.json()
        usd_rate = data['Valute']['USD']['Value']
        rate_date = data.get('Date', 'н/д')
        rate_source = "резервное зеркало cbr-xml-daily.ru"
        print(f"Курс USD получен с {rate_source}: {usd_rate:.4f} (дата: {rate_date})")
    except Exception as e_fallback:
        # оба источника недоступны — явно останавливаем выполнение,
        # чтобы не подставлять "угаданный" курс молча
        raise RuntimeError(
            "Не удалось получить курс валют ни с одного источника.\n"
            f"Ошибка основного API: {e_primary}\n"
            f"Ошибка резервного API: {e_fallback}"
        )

Запрошена дата: 15/09/2026 (вчера)
Курс USD, действовавший на эту дату (с официальное API ЦБ РФ (XML_daily.asp)): 84.3363
  Дата начала действия этого курса по данным ЦБ: 15.09.2026


## 2. Загрузка и обзор исходных данных

In [2]:
contracts = pd.read_excel('CASE_CONTRACTS.xlsx')
clients = pd.read_excel('CASE_CLIENTS.xlsx')
losses = pd.read_excel('CASE_LOSSES.xlsx')

print("--- Договоры (Contracts) ---")
display(contracts.head(3))
print(contracts.shape)

print("\n--- Клиенты (Clients) ---")
display(clients.head(3))
print(clients.shape)

print("\n--- Убытки (Losses) ---")
display(losses.head(3))
print(losses.shape)

--- Договоры (Contracts) ---


,contract_id,contract_num,product_name,client_id,contract_status,currency_name,duration,country,price,insurance_amount
0,21111219,ТТЕ7227715*****,Страхование путешественников,10161870404,Действует,Российский рубль,10,Беларусь,1096,2000000
1,21111381,ТТЕ7227715*****,Страхование путешественников,1017159879,Действует,Российский рубль,7,Индонезия,1918,5000000
2,21112353,БАДАМСТЕ55*****,Страхование путешественников,10161883357,Действует,Российский рубль,10,Беларусь,1096,2000000


(3711, 10)

--- Клиенты (Clients) ---


,client_id,last_name,first_name,middle_name,age,sex
0,10000041307,З*****,Р*****,О*****,30,F
1,10000133158,И*****,М*****,Ю*****,37,M
2,100003773,Е*****,О*****,С*****,50,M


(3711, 6)

--- Убытки (Losses) ---


,loss_id,client_id,loss_name,loss_payout_amt
0,1,10000713193,Оказание медицинской помощи,100000
1,2,100014656,Оказание медицинской помощи,50000
2,3,10006545463,Оказание медицинской помощи,30000


(45, 4)


,loss_id,client_id,loss_name,loss_payout_amt
0,1,10000713193,Оказание медицинской помощи,100000
1,2,100014656,Оказание медицинской помощи,50000
2,3,10006545463,Оказание медицинской помощи,30000


(45, 4)


## 3. Действующие полисы + атрибуты клиента

Оставляем только договоры со статусом `Действует` и присоединяем к ним инициалы (Ф. И. О.), пол и возраст клиента.

In [3]:
active_contracts = contracts[contracts['contract_status'] == 'Действует'].copy()

# проверка целостности справочника клиентов: client_id должен быть уникальным,
# иначе merge "размножит" строки договоров
assert clients['client_id'].is_unique, "В CASE_CLIENTS.xlsx есть дублирующиеся client_id"


def make_fio(row):
    """Собирает инициалы клиента в формате 'Ф. И. О.'"""
    parts = []
    for col in ('last_name', 'first_name', 'middle_name'):
        val = row[col]
        if pd.notna(val) and str(val).strip():
            parts.append(str(val).strip()[0].upper())
    return '. '.join(parts) + '.' if parts else ''


clients = clients.copy()
clients['fio'] = clients.apply(make_fio, axis=1)

n_before = len(active_contracts)
df = pd.merge(
    active_contracts,
    clients[['client_id', 'fio', 'age', 'sex']],
    on='client_id',
    how='left'
)
assert len(df) == n_before, "После merge с клиентами число строк изменилось — проверьте дубликаты client_id"

print(f"Действующих полисов: {len(df)}")
display(df[['contract_id', 'client_id', 'fio', 'sex', 'age']].head(3))

Действующих полисов: 3315


,contract_id,client_id,fio,sex,age
0,21111219,10161870404,А. Х. А.,M,20
1,21111381,1017159879,З. Х. М.,M,55
2,21112353,10161883357,Б. В. Щ.,F,61


## 4. Присоединение выплат по убыткам

В `CASE_LOSSES.xlsx` убыток привязан к клиенту, а не к конкретному договору, поэтому суммируем выплаты по `client_id` и присоединяем сумму к каждому действующему полису этого клиента. У клиентов из выборки нет более одного действующего полиса одновременно, поэтому дублирования сумм не возникает.

In [4]:
losses_agg = losses.groupby('client_id', as_index=False)['loss_payout_amt'].sum()
df = pd.merge(df, losses_agg, on='client_id', how='left')
df['loss_payout_amt'] = df['loss_payout_amt'].fillna(0)

print(f"Клиентов с выплатами по убыткам среди действующих полисов: {(df['loss_payout_amt'] > 0).sum()}")
display(df[df['loss_payout_amt'] > 0][['contract_id', 'client_id', 'fio', 'loss_payout_amt']].head(3))

Клиентов с выплатами по убыткам среди действующих полисов: 45


,contract_id,client_id,fio,loss_payout_amt
16,211195621,10000713193,A. A. P.,100000.0
32,211323787,100014656,А. Р. Э.,50000.0
90,2115957945,1000579095,Г. В. Я.,20000.0


## 5. Перевод сумм в USD и формирование итоговой витрины

Если полис уже оформлен в долларах США (`currency_name == 'Доллар США'`), курс считается актуальным и конвертация не требуется (по условию задания). Эта же логика применяется не только к цене полиса и страховой сумме, но и к сумме выплаты по убытку (валюта убытка в данных не указана, считаем её равной валюте договора).

In [5]:
def to_usd(value, currency_name):
    if currency_name == 'Доллар США':
        return value
    return value / usd_rate

df['price_usd'] = df.apply(lambda r: to_usd(r['price'], r['currency_name']), axis=1)
df['insurance_amount_usd'] = df.apply(lambda r: to_usd(r['insurance_amount'], r['currency_name']), axis=1)
# В CASE_LOSSES.xlsx нет столбца валюты, считаем, что выплата по убытку
# сделана в той же валюте, что и сам договор (currency_name), и применяем
# ту же логику конвертации, что и для цены полиса / страховой суммы.
df['loss_payout_usd'] = df.apply(lambda r: to_usd(r['loss_payout_amt'], r['currency_name']), axis=1)

datamart = df[[
    'contract_id', 'contract_num', 'product_name',
    'client_id', 'fio', 'sex', 'age',
    'contract_status', 'currency_name', 'duration', 'country',
    'price', 'price_usd',
    'insurance_amount', 'insurance_amount_usd',
    'loss_payout_amt', 'loss_payout_usd'
]].copy()

datamart.columns = [
    'ID договора', 'Номер договора', 'Продукт',
    'ID клиента', 'Ф. И. О.', 'Пол', 'Возраст',
    'Статус договора', 'Валюта', 'Срок (дни)', 'Страна',
    'Цена полиса', 'Цена полиса (USD)',
    'Страховая сумма', 'Страховая сумма (USD)',
    'Сумма выплат по убыткам', 'Сумма выплат по убыткам (USD)'
]

for col in ['Цена полиса (USD)', 'Страховая сумма (USD)', 'Сумма выплат по убыткам (USD)']:
    datamart[col] = datamart[col].round(2)

print(f"Итоговая витрина: {len(datamart)} строк, {datamart.shape[1]} столбцов")
display(datamart.head(3))

Итоговая витрина: 3315 строк, 17 столбцов


,ID договора,Номер договора,Продукт,ID клиента,Ф. И. О.,Пол,Возраст,Статус договора,Валюта,Срок (дни),Страна,Цена полиса,Цена полиса (USD),Страховая сумма,Страховая сумма (USD),Сумма выплат по убыткам,Сумма выплат по убыткам (USD)
0,21111219,ТТЕ7227715*****,Страхование путешественников,10161870404,А. Х. А.,M,20,Действует,Российский рубль,10,Беларусь,1096,13.00,2000000,23714.58,0.0,0.0
1,21111381,ТТЕ7227715*****,Страхование путешественников,1017159879,З. Х. М.,M,55,Действует,Российский рубль,7,Индонезия,1918,22.74,5000000,59286.45,0.0,0.0
2,21112353,БАДАМСТЕ55*****,Страхование путешественников,10161883357,Б. В. Щ.,F,61,Действует,Российский рубль,10,Беларусь,1096,13.00,2000000,23714.58,0.0,0.0


## 6. Сохранение в Excel с оформлением

In [6]:
output_file = 'insurance_datamart.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    datamart.to_excel(writer, sheet_name='Витрина действующих полисов', index=False)

wb = openpyxl.load_workbook(output_file)
ws = wb['Витрина действующих полисов']

header_fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
row_font = Font(name='Calibri', size=10)
zebra_fill = PatternFill(start_color='F2F4F8', end_color='F2F4F8', fill_type='solid')
white_fill = PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')
thin_border = Border(
    left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'),
    top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9')
)

numeric_cols = {1, 4, 7, 10}          # ID договора, ID клиента, Возраст, Срок
category_cols = {6, 8, 9, 11}         # Пол, Статус, Валюта, Страна
money_cols = {12, 13, 14, 15, 16, 17} # все суммы

ws.row_dimensions[1].height = 28
for col in range(1, ws.max_column + 1):
    cell = ws.cell(row=1, column=col)
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

headers = [ws.cell(row=1, column=c).value for c in range(1, ws.max_column + 1)]

for row in range(2, ws.max_row + 1):
    ws.row_dimensions[row].height = 20
    is_zebra = (row % 2 == 0)
    for col in range(1, ws.max_column + 1):
        cell = ws.cell(row=row, column=col)
        cell.font = row_font
        cell.border = thin_border
        cell.fill = zebra_fill if is_zebra else white_fill

        if col in numeric_cols or col in category_cols:
            cell.alignment = Alignment(horizontal='center', vertical='center')
        elif col in money_cols:
            cell.alignment = Alignment(horizontal='right', vertical='center')
            if isinstance(cell.value, (int, float)):
                cell.number_format = '#,##0.00' if 'USD' in headers[col - 1] else '#,##0'
        else:
            cell.alignment = Alignment(horizontal='left', vertical='center')

for col_cells in ws.columns:
    max_len = max(len(str(c.value or '')) for c in col_cells)
    col_letter = get_column_letter(col_cells[0].column)
    ws.column_dimensions[col_letter].width = max(max_len + 3, 12)

ws.auto_filter.ref = ws.dimensions
wb.save(output_file)

print(f"Курс использован: {usd_rate:.4f} ({rate_source}, действует с {rate_date})")
print(f"Готово! Витрина сохранена в файл '{output_file}', строк: {len(datamart)}")

Курс использован: 84.3363 (официальное API ЦБ РФ (XML_daily.asp), действует с 15.09.2026)
Готово! Витрина сохранена в файл 'insurance_datamart.xlsx', строк: 3315
